# Question 3 — Knowledge Distillation (worked numerically)
Zaid Bin Naveed · 23K-0028

Vocabulary: `[hospital, clinic, ward, school]`, label **y = [1, 0, 0, 0]**.
Loss: $L = \alpha T^2\,\mathrm{KL}(p_T \,\|\, q_T) + (1-\alpha)\,\mathrm{CE}(y, q_1)$ with $T=2$, $\alpha=0.5$, natural log throughout.

In [1]:
import json
from pathlib import Path

import numpy as np

np.set_printoptions(precision=6, suppress=True)

z_t = np.array([5.0, 3.0, 2.0, 0.0])
z_s = np.array([3.0, 2.5, 1.0, 0.5])
y = np.array([1.0, 0.0, 0.0, 0.0])
T, alpha, eta = 2.0, 0.5, 0.5
words = ["hospital", "clinic", "ward", "school"]

## (a) Probabilities at T = 1
$\mathrm{softmax}(z/T)_i = \dfrac{e^{z_i/T}}{\sum_j e^{z_j/T}}$

In [2]:
def softmax(z, temp=1.0):
    e = np.exp(z / temp)
    return e / e.sum()

p1, q1 = softmax(z_t), softmax(z_s)
print("exp(z_t) =", np.exp(z_t), " sum =", np.exp(z_t).sum())
print("exp(z_s) =", np.exp(z_s), " sum =", np.exp(z_s).sum())
print("p1 (teacher) =", p1)
print("q1 (student) =", q1)

exp(z_t) = [148.413159  20.085537   7.389056   1.      ]  sum = 176.8877521246949
exp(z_s) = [20.085537 12.182494  2.718282  1.648721]  sum = 36.63503398305032
p1 (teacher) = [0.839025 0.11355  0.041773 0.005653]
q1 (student) = [0.54826  0.332537 0.074199 0.045004]


## (b) Softened probabilities at T = 2

In [3]:
pT, qT = softmax(z_t, T), softmax(z_s, T)
print("exp(z_t/2) =", np.exp(z_t / T), " sum =", np.exp(z_t / T).sum())
print("exp(z_s/2) =", np.exp(z_s / T), " sum =", np.exp(z_s / T).sum())
print("pT (teacher) =", pT)
print("qT (student) =", qT)

exp(z_t/2) = [12.182494  4.481689  2.718282  1.      ]  sum = 20.382464859500583
exp(z_s/2) = [4.481689 3.490343 1.648721 1.284025]  sum = 10.904778715187776
pT (teacher) = [0.597695 0.21988  0.133364 0.049062]
qT (student) = [0.410984 0.320075 0.151193 0.117749]


## (c) KL divergence
$\mathrm{KL}(p_T\|q_T) = \sum_i p_{T,i}\,\ln\dfrac{p_{T,i}}{q_{T,i}}$

In [4]:
kl_terms = pT * np.log(pT / qT)
kl = kl_terms.sum()
print("ln(pT/qT)   =", np.log(pT / qT))
print("terms       =", kl_terms)
print("KL(pT||qT)  =", round(kl, 6))

ln(pT/qT)   = [ 0.374526 -0.375474 -0.125474 -0.875474]
terms       = [ 0.223852 -0.082559 -0.016734 -0.042952]
KL(pT||qT)  = 0.081607


## (d) Hard-label cross-entropy
$\mathrm{CE}(y, q_1) = -\sum_i y_i \ln q_{1,i} = -\ln q_{1,\text{hospital}}$

In [5]:
ce = -np.sum(y * np.log(q1))
print("CE =", round(ce, 6))

CE = 0.601005


## (e) Total distillation loss

In [6]:
def total_loss(z_student):
    kl_ = np.sum(pT * np.log(pT / softmax(z_student, T)))
    ce_ = -np.sum(y * np.log(softmax(z_student)))
    return alpha * T**2 * kl_ + (1 - alpha) * ce_, kl_, ce_

L0 = total_loss(z_s)[0]
print(f"distillation term = {alpha} * {T**2} * {kl:.6f} = {alpha * T**2 * kl:.6f}")
print(f"hard-label term   = {1 - alpha} * {ce:.6f} = {(1 - alpha) * ce:.6f}")
print(f"L = {L0:.6f}")

distillation term = 0.5 * 4.0 * 0.081607 = 0.163215
hard-label term   = 0.5 * 0.601005 = 0.300502
L = 0.463717


## (f) Gradient with respect to the student logits
$\dfrac{\partial\,\mathrm{KL}}{\partial z_s} = \dfrac{1}{T}(q_T - p_T)$ and $\dfrac{\partial\,\mathrm{CE}}{\partial z_s} = q_1 - y$, so

$\nabla_{z_s} L = \alpha T^2 \cdot \dfrac{1}{T}(q_T - p_T) + (1-\alpha)(q_1 - y) = \alpha T (q_T - p_T) + (1-\alpha)(q_1 - y)$

A central finite-difference check confirms the analytic gradient.

In [7]:
g_soft = alpha * T * (qT - pT)
g_hard = (1 - alpha) * (q1 - y)
grad = g_soft + g_hard

h = 1e-6
grad_fd = np.array([(total_loss(z_s + h * e)[0] - total_loss(z_s - h * e)[0]) / (2 * h) for e in np.eye(4)])
print("soft part  =", g_soft)
print("hard part  =", g_hard)
print("gradient   =", grad)
print("finite diff=", grad_fd, " max |diff| =", np.abs(grad - grad_fd).max())
print("sum of gradient components =", round(grad.sum(), 12))

soft part  = [-0.186711  0.100195  0.017829  0.068687]
hard part  = [-0.22587   0.166268  0.037099  0.022502]
gradient   = [-0.412581  0.266463  0.054928  0.091189]
finite diff= [-0.412581  0.266463  0.054928  0.091189]  max |diff| = 1.9252188732110653e-10
sum of gradient components = -0.0


## (g) One gradient-descent step, η = 0.5
$z_s' = z_s - \eta\,\nabla_{z_s} L$

In [8]:
z_new = z_s - eta * grad
L1, kl1, ce1 = total_loss(z_new)
print("new logits =", z_new)
print("new qT     =", softmax(z_new, T))
print("new q1     =", softmax(z_new))
print(f"new KL = {kl1:.6f}, new CE = {ce1:.6f}")
print(f"new L  = {L1:.6f}  (was {L0:.6f}, change {L1 - L0:+.6f})")

new logits = [3.20629  2.366768 0.972536 0.454405]
new qT     = [0.447006 0.293774 0.146305 0.112914]
new q1     = [0.623889 0.269468 0.066834 0.039809]
new KL = 0.056682, new CE = 0.471783
new L  = 0.349256  (was 0.463717, change -0.114461)


In [9]:
results = {
    "p1": p1, "q1": q1, "pT": pT, "qT": qT, "kl_terms": kl_terms, "kl": kl, "ce": ce, "loss": L0,
    "grad_soft": g_soft, "grad_hard": g_hard, "grad": grad, "z_new": z_new,
    "qT_new": softmax(z_new, T), "q1_new": softmax(z_new), "kl_new": kl1, "ce_new": ce1, "loss_new": L1,
}
Path("results").mkdir(exist_ok=True)
Path("results/q3_results.json").write_text(json.dumps({k: np.round(v, 10).tolist() for k, v in results.items()}, indent=2))
print("saved results/q3_results.json")

saved results/q3_results.json
